In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import sklearn
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("polars:", pl.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("shap:", shap.__version__)

print("\nAmbiente configurado com sucesso!")

In [ ]:
df = pd.read_csv('../data/raw/application_train.csv')

print("Shape:", df.shape)
print("\nTaxa de default (TARGET):")
print(df['TARGET'].value_counts(normalize=True))

## Dataset Overview

The dataset contains 307,511 loan applications with 122 features.
The default rate (TARGET=1) is 8%, reflecting Home Credit's profile:
a consumer finance provider targeting borrowers with little or no formal
credit history, combining higher-risk borrower profiles with real estate collateral.

The class imbalance (92% vs 8%) makes accuracy an inadequate metric.
Primary evaluation metrics: AUC-ROC, KS statistic, Gini coefficient.

Case Description:

Many people struggle to get loans due to insufficient or non-existent credit histories. And, unfortunately, this population is often taken advantage of by untrustworthy lenders.

Home Credit Group

Home Credit strives to broaden financial inclusion for the unbanked population by providing a positive and safe borrowing experience. In order to make sure this underserved population has a positive loan experience, Home Credit makes use of a variety of alternative data--including telco and transactional information--to predict their clients' repayment abilities.

While Home Credit is currently using various statistical and machine learning methods to make these predictions, they're challenging Kagglers to help them unlock the full potential of their data. Doing so will ensure that clients capable of repayment are not rejected and that loans are given with a principal, maturity, and repayment calendar that will empower their clients to be successful.

In [ ]:
# Variable types and missing values overview
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

summary = pd.DataFrame({
    'dtype': df.dtypes,
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

print("=== Variable Types ===")
print(df.dtypes.value_counts())

print("\n=== Top 20 variables with most missing values ===")
print(summary[summary['missing_count'] > 0].head(20))

print(f"\nTotal variables with missing values: {(missing > 0).sum()}")
print(f"Total variables with >50% missing: {(missing_pct > 50).sum()}")

In [ ]:
# Separate numeric and categorical variables
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['str']).columns.tolist()

# Remove TARGET from numeric list
numeric_cols = [c for c in numeric_cols if c != 'TARGET']

print(f"Numeric variables: {len(numeric_cols)}")
print(f"Categorical variables: {len(categorical_cols)}")

print("\n=== Categorical variables and unique values ===")
for col in categorical_cols:
    n_unique = df[col].nunique()
    top_values = df[col].value_counts().head(3).to_dict()
    print(f"\n{col} ({n_unique} unique values)")
    print(f"  Top 3: {top_values}")

In [ ]:
desc = pd.read_csv('../data/raw/HomeCredit_columns_description.csv', 
                   encoding='latin-1')
print(desc.shape)
print(desc.head())
print(desc.columns.tolist())

In [ ]:
# Filter descriptions for main application table only
app_desc = desc[desc['Table'] == 'application_{train|test}.csv'].copy()
app_desc = app_desc[['Row', 'Description', 'Special']].reset_index(drop=True)

print(f"Variables described: {len(app_desc)}")
print("\n=== Full variable dictionary ===")
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)
print(app_desc)

In [ ]:
# Identify numeric variables with at least one negative value
neg_vars = df[numeric_cols].lt(0).any()
neg_vars = neg_vars[neg_vars].index.tolist()

summary_neg = df[neg_vars].agg(['min', 'max', 'mean']).T
summary_neg['neg_count'] = df[neg_vars].lt(0).sum()

print(f"Total de variáveis numéricas com valores negativos: {len(neg_vars)}")
print(summary_neg.round(2))

In [ ]:
df.loc[df['DAYS_EMPLOYED'] == 365243, 'NAME_INCOME_TYPE'].value_counts()

In [ ]:
# in variable 'DAYS_EMPLOYED' we observe 252,135 negative values over 307,511 = 55,376
# comparing to 'NAME_INCOME_TYPE' we find 55,374. Will not further investigate the other two cases, because is a too small difference.

In [ ]:
# DAYS_EMPLOYED has a sentinel value (365243) representing "not applicable",
# used mainly for Pensioner and Unemployed clients who have no current job start date.
# Strategy: flag first, then convert sentinel to NaN. XGBoost handles NaN natively;
# imputation strategy for the logistic regression scorecard will be revisited during
# feature engineering (likely WoE binning, which treats missing as its own bin).

df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype('int8')
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

print(df['DAYS_EMPLOYED_ANOM'].value_counts())
print(df['DAYS_EMPLOYED'].isna().sum())

In [ ]:
df.shape

In [ ]:
# just created our first new variable on the cell above

In [ ]:
# Convert DAYS_* variables to positive, human-readable units, keeping the
# original Kaggle columns untouched for traceability. This is purely an
# interpretability transformation: it does not affect model performance,
# since tree-based splits are invariant to sign, and coefficient sign in
# logistic regression simply flips accordingly.

df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).round(1)
df['YEARS_REGISTRATION'] = (-df['DAYS_REGISTRATION'] / 365).round(1)
df['YEARS_ID_PUBLISH'] = (-df['DAYS_ID_PUBLISH'] / 365).round(1)
df['YEARS_LAST_PHONE_CHANGE'] = (-df['DAYS_LAST_PHONE_CHANGE'] / 365).round(1)

print(df[['DAYS_BIRTH', 'AGE_YEARS',
          'DAYS_REGISTRATION', 'YEARS_REGISTRATION',
          'DAYS_ID_PUBLISH', 'YEARS_ID_PUBLISH',
          'DAYS_LAST_PHONE_CHANGE', 'YEARS_LAST_PHONE_CHANGE']].head())

In [ ]:
# Same readability transformation applied to the remaining DAYS_* variable.
# NaN values (former 365243 sentinel) propagate correctly through the division.
df['YEARS_EMPLOYED'] = (-df['DAYS_EMPLOYED'] / 365).round(1)

In [ ]:
# Correlation of numeric variables with TARGET
correlations = df[numeric_cols + ['TARGET']].corr()['TARGET'].drop('TARGET')
correlations = correlations.abs().sort_values(ascending=False)

print("=== Top 20 variables by absolute correlation with TARGET ===")
print(correlations.head(20).round(4))

print("\n=== Bottom 10 (lowest correlation) ===")
print(correlations.tail(10).round(4))

In [ ]:
pearson_corr = df[numeric_cols + ['TARGET']].corr(method='pearson')['TARGET'].drop('TARGET')
spearman_corr = df[numeric_cols + ['TARGET']].corr(method='spearman')['TARGET'].drop('TARGET')

comparison = pd.DataFrame({
    'pearson': pearson_corr,
    'spearman': spearman_corr,
    'gap': (spearman_corr.abs() - pearson_corr.abs())
}).sort_values('gap', ascending=False)

print("=== Variáveis onde Spearman > Pearson (indício de não-linearidade) ===")
print(comparison.head(15).round(4))

In [ ]:
# reset numeric cols because we we created new numeric variables
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'TARGET']

In [ ]:
# Correlation of numeric variables with TARGET
correlations = df[numeric_cols + ['TARGET']].corr()['TARGET'].drop('TARGET')
correlations = correlations.abs().sort_values(ascending=False)

print("=== Top 20 variables by absolute correlation with TARGET ===")
print(correlations.head(20).round(4))

print("\n=== Bottom 10 (lowest correlation) ===")
print(correlations.tail(10).round(4))

In [ ]:
pearson_corr = df[numeric_cols + ['TARGET']].corr(method='pearson')['TARGET'].drop('TARGET')
spearman_corr = df[numeric_cols + ['TARGET']].corr(method='spearman')['TARGET'].drop('TARGET')

comparison = pd.DataFrame({
    'pearson': pearson_corr,
    'spearman': spearman_corr,
    'gap': (spearman_corr.abs() - pearson_corr.abs())
}).sort_values('gap', ascending=False)

print("=== Variáveis onde Spearman > Pearson (indício de não-linearidade) ===")
print(comparison.head(15).round(4))

In [ ]:
df.shape

In [ ]:
# Defragment the DataFrame after multiple individual column insertions
# during the negative-values treatment step. This resolves the
# PerformanceWarning without changing any data.
df = df.copy()

In [ ]:
df.head(2)

In [ ]:
df[['TARGET','EXT_SOURCE_3','EXT_SOURCE_2','EXT_SOURCE_1','DAYS_BIRTH']].head(5)

In [ ]:
# Default rate by decile for EXT_SOURCE_1/2/3 and DAYS_BIRTH.
# Uses qcut (quantile-based) instead of cut (fixed-width) to ensure
# roughly equal-sized groups, which is what "decile" requires.

for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH']:
    decile_col = f'{col}_DECILE'
    df[decile_col] = pd.qcut(df[col], q=10, labels=False, duplicates='drop')
    default_by_decile = df.groupby(decile_col)['TARGET'].mean()
    print(f'=== Default rate by decile: {col} ===')
    print(default_by_decile.round(4))
    print()

In [ ]:
# Confirm what each DAYS_BIRTH decile actually contains,
# using AGE_YEARS for readability (positive values, easier to interpret).
age_by_decile = df.groupby('DAYS_BIRTH_DECILE')['AGE_YEARS'].agg(['min', 'max', 'mean'])
print(age_by_decile.round(1))

In [ ]:
# For each variable with missing values, compare default rate between
# missing and non-missing groups. This does not create permanent columns;
# the missing mask is computed on the fly inside the loop for diagnostic purposes only.

missing_vars = df.columns[df.isnull().any()].tolist()

results = []
for col in missing_vars:
    is_missing = df[col].isnull()
    rate_missing = df.loc[is_missing, 'TARGET'].mean()
    rate_not_missing = df.loc[~is_missing, 'TARGET'].mean()
    diff = rate_missing - rate_not_missing
    results.append({
        'variable': col,
        'missing_count': is_missing.sum(),
        'default_rate_missing': rate_missing,
        'default_rate_not_missing': rate_not_missing,
        'diff': diff
    })

missing_target_df = pd.DataFrame(results).sort_values('diff', key=abs, ascending=False)
print(missing_target_df.round(4).to_string(index=False))

In [ ]:
# Missing pattern vs TARGET, excluding decile columns (artifacts of prior
# transformation, not independent business variables) and excluding variables
# with very small missing counts, where the default rate is statistically
# unstable and not a reliable signal.

MIN_MISSING_COUNT = 500

missing_vars = [c for c in df.columns[df.isnull().any()].tolist() if not c.endswith('_DECILE')]

results = []
for col in missing_vars:
    is_missing = df[col].isnull()
    if is_missing.sum() < MIN_MISSING_COUNT:
        continue
    rate_missing = df.loc[is_missing, 'TARGET'].mean()
    rate_not_missing = df.loc[~is_missing, 'TARGET'].mean()
    diff = rate_missing - rate_not_missing
    results.append({
        'variable': col,
        'missing_count': is_missing.sum(),
        'default_rate_missing': rate_missing,
        'default_rate_not_missing': rate_not_missing,
        'diff': diff
    })

missing_target_df = pd.DataFrame(results).sort_values('diff', key=abs, ascending=False)
print(missing_target_df.round(4).to_string(index=False))

In [ ]:
df.head(2)